In [2]:
"""
Two-step synthetic narrative generation for structural similarity evaluation.

Step 1: Ask GPT-5.4 for a batch of diverse (theme, core_events) specifications.
Step 2: For each specification, ask GPT-5.4 to generate 3 stories (anchor,
        same-structure variant, different-structure variant) following the
        eventful-sentence-friendly formatting constraints.

Output: a CSV with 100 rows (one per theme), each containing the theme,
core events, and the three generated stories, saved to:
/scratch/shayan/Projects/NarrativeSimilarity/data/eval_data/synthetic_v2_anchor_eval_data.csv
"""

import json
import csv
import time
from pathlib import Path

from openai import OpenAI
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
API_KEY_PATH = "/scratch/shayan/Projects/NarrativeSimilarity/openai_key.txt"
OUTPUT_PATH = "/scratch/shayan/Projects/NarrativeSimilarity/data/eval_data/synthetic_v2_anchor_eval_data.csv"
MODEL = "gpt-5.4"

N_THEMES_TOTAL = 100
THEMES_PER_BATCH = 20  # ask for themes in batches to keep outputs diverse and manageable
MAX_RETRIES = 3
SLEEP_BETWEEN_CALLS = 1.0  # seconds, be polite to rate limits

# ---------------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------------
api_key = Path(API_KEY_PATH).read_text().strip()
client = OpenAI(api_key=api_key)

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)


def call_model(system_prompt: str, user_prompt: str, expect_json: bool = True):
    """Call the model with retries, returning parsed JSON (or raw text)."""
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                response_format={"type": "json_object"} if expect_json else None,
            )
            content = response.choices[0].message.content
            if expect_json:
                return json.loads(content)
            return content
        except Exception as e:  # noqa: BLE001 - broad on purpose for retry loop
            last_err = e
            time.sleep(SLEEP_BETWEEN_CALLS * (attempt + 1))
    raise RuntimeError(f"Model call failed after {MAX_RETRIES} attempts: {last_err}")


# ---------------------------------------------------------------------------
# Step 1: generate (theme, core_events) specifications
# ---------------------------------------------------------------------------
STEP1_SYSTEM_PROMPT = """You are generating specifications for synthetic narrative \
data used in research on narrative structural similarity."""

STEP1_USER_PROMPT_TEMPLATE = """Generate {n} distinct narrative specifications for a \
personal-narrative-style dataset. Each specification consists of:
- a short "theme" (1 sentence describing the overall scenario, e.g. "a person \
training for and completing their first marathon")
- 4 or 5 "core_events" that must occur in the story, listed in a natural \
chronological order (each a short phrase, e.g. "signs up for the race")

Requirements:
- Themes should span a wide variety of everyday personal experiences (career, \
relationships, travel, health, family, hobbies, education, everyday problems, etc.).
- Avoid repeating similar themes across specifications.
- Core events should be concrete and specific enough to be reordered or altered \
later (e.g. omitted, reordered, or given a different outcome) while still making \
sense as a narrative skeleton.
- Do not write full stories here, only the theme and the list of core events.

Return a JSON object with a single key "specifications", containing a list of \
{n} objects, each with keys "theme" and "core_events" (a list of 4-5 strings).
"""


def generate_theme_specs(n_total: int, batch_size: int):
    specs = []
    pbar = tqdm(total=n_total, desc="Step 1: theme specs")
    while len(specs) < n_total:
        n = min(batch_size, n_total - len(specs))
        result = call_model(
            STEP1_SYSTEM_PROMPT,
            STEP1_USER_PROMPT_TEMPLATE.format(n=n),
            expect_json=True,
        )
        batch_specs = result["specifications"]
        specs.extend(batch_specs)
        pbar.update(len(batch_specs))
        time.sleep(SLEEP_BETWEEN_CALLS)
    pbar.close()
    return specs[:n_total]


# ---------------------------------------------------------------------------
# Step 2: generate 3 stories per (theme, core_events) specification
# ---------------------------------------------------------------------------
STEP2_SYSTEM_PROMPT = """You are generating synthetic narrative data for research \
on narrative similarity."""

STEP2_USER_PROMPT_TEMPLATE = """Generate 3 short chronological narratives for the \
following theme and set of core events.

Theme: {theme}
Core events (in original order): {core_events}

Requirements:
- All stories should describe the same underlying theme.
- All stories must naturally include all core events.
- The narratives should remain realistic, coherent, and chronological.
- Avoid copying phrases directly across stories.
- Use simple and explicit event descriptions.
- Prefer one main event per sentence.
- Do not combine multiple core events into a single sentence.
- Each core event should appear clearly and separately in the narrative.
- Keep sentences short and easy to align across stories.

Story roles:
- Story 1: the anchor story.
- Story 2: the most similar to Story 1 in event flow and structure.
- Story 3: still related to Story 1, but with more noticeable differences in \
event emphasis and ordering (e.g. reordered events, omitted events, altered \
causal links, or a different outcome).

Return a JSON object with keys "story_1", "story_2", and "story_3", each a \
single string containing the full narrative (multiple sentences).
"""


def generate_stories_for_spec(spec: dict):
    theme = spec["theme"]
    core_events = spec["core_events"]
    result = call_model(
        STEP2_SYSTEM_PROMPT,
        STEP2_USER_PROMPT_TEMPLATE.format(
            theme=theme,
            core_events="; ".join(core_events),
        ),
        expect_json=True,
    )
    return result


# ---------------------------------------------------------------------------
# Run pipeline
# ---------------------------------------------------------------------------
theme_specs = generate_theme_specs(N_THEMES_TOTAL, THEMES_PER_BATCH)

rows = []
for i, spec in enumerate(tqdm(theme_specs, desc="Step 2: generating stories")):
    try:
        stories = generate_stories_for_spec(spec)
        rows.append(
            {
                "example_id": i,
                "theme": spec["theme"],
                "core_events": json.dumps(spec["core_events"]),
                "story_1_anchor": stories["story_1"],
                "story_2_similar": stories["story_2"],
                "story_3_dissimilar": stories["story_3"],
            }
        )
    except Exception as e:  # noqa: BLE001
        tqdm.write(f"  FAILED on example {i} ({spec.get('theme', '?')}): {e}")
    time.sleep(SLEEP_BETWEEN_CALLS)

# ---------------------------------------------------------------------------
# Save to CSV
# ---------------------------------------------------------------------------
fieldnames = [
    "example_id",
    "theme",
    "core_events",
    "story_1_anchor",
    "story_2_similar",
    "story_3_dissimilar",
]
with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} examples to {OUTPUT_PATH}")

Step 2: generating stories: 100%|██████████| 100/100 [12:37<00:00,  7.58s/it]

Saved 100 examples to /scratch/shayan/Projects/NarrativeSimilarity/data/eval_data/synthetic_v2_anchor_eval_data.csv
